# 22 · EDA y visualización para Data Science

Exploratory Data Analysis no es 'hacer gráficos bonitos': es entender cómo se generaron los datos, qué puede salir mal y qué preguntas vale la pena modelar.

## Objetivos
- Auditar distribución, missing, outliers y cardinalidad.
- Explorar relaciones uni/bi/multivariadas sin target leakage.
- Visualizar incertidumbre y tamaños de muestra.
- Detectar Simpson's paradox y segmentación oculta.
- Crear un reporte reproducible.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED=42; rng=np.random.default_rng(SEED); n=3000
df=pd.DataFrame({'edad':rng.normal(45,14,n).clip(18,90),'ingreso':rng.lognormal(10.5,.7,n),'region':rng.choice(['N','C','S'],n,p=[.2,.55,.25]),'canal':rng.choice(['web','presencial','telefono'],n)})
df['score']=.04*df.edad+.000015*df.ingreso+(df.region=='C')*.5+rng.normal(0,1,n); df.loc[rng.choice(n,120,replace=False),'ingreso']=np.nan
df.head()

## 1. Perfil de tabla
Preguntas mínimas: ¿qué representa una fila?, ¿qué periodo cubre?, ¿qué unidad de análisis?, ¿hay duplicados lógicos?, ¿cómo se creó cada variable?, ¿qué tan reciente es?, ¿puede existir en inferencia?


In [ ]:
profile=pd.DataFrame({'dtype':df.dtypes.astype(str),'missing_n':df.isna().sum(),'missing_pct':df.isna().mean(),'unique':df.nunique(dropna=False)}); display(profile); print('duplicados exactos',df.duplicated().sum())

## 2. Distribuciones y escala
Media/SD no bastan para colas largas. Compara mediana, IQR, percentiles y escala log. Boxplots son útiles, pero una observación fuera de 1.5×IQR no es automáticamente 'error'.


In [ ]:
fig,ax=plt.subplots(1,3,figsize=(14,4)); df.edad.hist(bins=35,ax=ax[0]); df.ingreso.hist(bins=40,ax=ax[1]); np.log1p(df.ingreso).hist(bins=40,ax=ax[2]); ax[0].set_title('edad'); ax[1].set_title('ingreso'); ax[2].set_title('log ingreso'); plt.show(); display(df[['edad','ingreso','score']].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

## 3. Categóricas y cardinalidad
Siempre muestra n además de porcentaje. Un 80% calculado sobre 5 observaciones es distinto a 80% sobre 50.000. Categorías nuevas en producción requieren estrategia (`unknown`, hashing, embeddings, reglas).


In [ ]:
for c in ['region','canal']: display(df[c].value_counts(dropna=False).to_frame('n').assign(pct=lambda z:z.n/z.n.sum()))

## 4. Relaciones
Scatter + alpha/hexbin revelan patrones continuos; box/violin comparan grupos; tablas cruzadas muestran composición. Correlación no captura todas las relaciones y no implica causalidad.


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].scatter(df.edad,df.score,alpha=.15,s=8); ax[0].set(xlabel='edad',ylabel='score'); df.boxplot(column='score',by='region',ax=ax[1]); plt.suptitle(''); plt.show(); print(df[['edad','ingreso','score']].corr(numeric_only=True).round(3))

## 5. Missingness como señal
Missing puede ser MCAR, MAR o MNAR. No siempre podemos distinguirlos con datos observados. Grafica tasa de missing por segmento/tiempo y pregunta por proceso de captura. Un missing sistemático por canal puede codificar información institucional.


In [ ]:
display(df.assign(ingreso_missing=df.ingreso.isna()).groupby(['region','canal']).ingreso_missing.agg(['mean','size']).sort_values('mean',ascending=False))

## 6. Visualización honesta
- ejes truncados pueden exagerar diferencias;
- mapas de color arcoíris distorsionan percepción;
- demasiadas categorías vuelven ilegible un gráfico;
- agregados esconden distribución;
- no muestres una tendencia sin tamaño de muestra o incertidumbre cuando importa.

## 7. EDA para ML
Busca: leakage, shift entre train/test, imbalance, grupos repetidos, timestamps, duplicados, proxy variables, labels ruidosos y features casi constantes.

## Herramientas
Pandas/Polars, Matplotlib, Plotly, Altair, ydata-profiling, Sweetviz, DuckDB, Great Expectations/Pandera.

## Ejercicios
1. Genera un perfil automático con `ydata-profiling`.
2. Crea un dataset con Simpson's paradox y visualízalo.
3. Grafica distribución train vs test por feature.
4. Implementa un reporte HTML simple con 10 checks.
5. Encuentra outliers con IQR, robust z y IsolationForest; compara.
6. Diseña un dashboard para monitorear datos mensualmente.
